# Text-Based Prediction of Book Review Popularity
### Machine Learning Mini-Project (UE24CS352A)
**Reference Paper**: Stanford CS229 — *Text-Based Prediction of Book Review Popularity* (Bridget Daly)  
**Hardware Accelerated**: NVIDIA CUDA (PyTorch + XGBoost GPU Hist)

---
### Project Goals & Syllabus Mapping:
1. **Core Problem**: Classify whether a Goodreads book review achieves high relative engagement ($	ext{like\_share} > 0.02$).
2. **Replication**: Compare Logistic Regression, XGBoost, and Deep Neural Networks across feature subsets (A: Metadata, B: Metadata + NLP, D: Metadata + NLP + TF-IDF).
3. **⭐ Official Add-On (Units 1 & 3)**: **Cost-Sensitive Loss Weighting (`scale_pos_weight` & `class_weight='balanced'`) vs. Random Undersampling**.
4. **🔬 Student Experimentation**:
   - **Experiment 1 (Unit 2)**: ANN Architecture Depth & Learning Rate Ablation.
   - **Experiment 2 (Unit 1)**: Decision Tree Depth vs. Empirical Bias-Variance Tradeoff Curve.


In [ ]:
import sys
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# GPU Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Hardware Acceleration Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

# Download required NLTK resources (compatible with both legacy and NLTK 3.9+)
for res in ['vader_lexicon', 'punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass


## Stage 1: Data Ingestion & Stratified Preprocessing
We stream the raw JSON reviews and books datasets from Kaggle input paths, filtering uninformative rows and creating the relative popularity target variable.


In [ ]:
# Kaggle Input Dataset Auto-Discovery & Path Resolution
print("🔍 Searching for attached datasets in /kaggle/input/...")
all_files = []
if os.path.exists("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            all_files.append(os.path.join(root, f))
    print(f"📁 Total files found in /kaggle/input: {len(all_files)}")
    for f in all_files:
        print(f"   • {f}")

def find_dataset(keyword, fallback):
    for f in all_files:
        fname = os.path.basename(f).lower()
        if keyword in fname and (fname.endswith('.json') or fname.endswith('.json.gz') or fname.endswith('.csv')):
            return f
    return fallback

review_path = find_dataset("review", "/kaggle/input/datasets/prekshacm/goodreads-reviews-dedup-json/goodreads_reviews_dedup.json")
book_path = find_dataset("book", "/kaggle/input/datasets/prekshacm/goodreads-books-json/goodreads_books.json")

print(f"
📂 Resolved Review Dataset Path: {review_path}")
print(f"📂 Resolved Book Dataset Path:   {book_path}")

import gzip
def smart_open(filepath):
    if filepath.endswith('.gz'):
        return gzip.open(filepath, 'rt', encoding='utf-8')
    return open(filepath, 'r', encoding='utf-8')

SAMPLE_LIMIT = 100000  # High-speed representative sample (<15 mins total)

print(f"⏳ Reading and parsing review data (cap: {SAMPLE_LIMIT:,} rows)...")
reviews = []
total_parsed = 0

with smart_open(review_path) as f:
    for line in f:
        total_parsed += 1
        entry = json.loads(line)
        # Keep reviews with at least 1 like or comment
        if entry.get('n_votes', 0) + entry.get('n_comments', 0) > 0:
            reviews.append({
                'user_id': entry['user_id'],
                'book_id': entry['book_id'],
                'user_rating': entry['rating'],
                'review_text': entry['review_text'],
                'date_added': entry['date_added'],
                'review_likes': entry['n_votes'] + entry['n_comments']
            })
        if len(reviews) >= SAMPLE_LIMIT:
            break

reviews_df = pd.DataFrame(reviews)
print(f"✅ Loaded {len(reviews_df):,} informative reviews from {total_parsed:,} parsed records.")

# Compute user total reviews
user_counts = reviews_df.groupby('user_id')['book_id'].count().rename('user_reviews')
reviews_df = reviews_df.merge(user_counts, on='user_id')

# Read Book Metadata
print("⏳ Reading book metadata...")
book_ratings = {}
with smart_open(book_path) as f:
    for line in f:
        entry = json.loads(line)
        b_id = entry['book_id']
        try:
            avg_r = float(entry.get('average_rating', 0.0))
            rev_cnt = int(entry.get('text_reviews_count', 0))
            if rev_cnt >= 10:
                book_ratings[b_id] = avg_r
        except:
            continue

books_df = pd.DataFrame(list(book_ratings.items()), columns=['book_id', 'avg_rating'])
merged_df = reviews_df.merge(books_df, on='book_id')

# Calculate total review likes per book
book_likes = merged_df.groupby('book_id')['review_likes'].sum().rename('book_review_likes')
merged_df = merged_df.merge(book_likes, on='book_id')

# Filter books with at least 10 total likes
merged_df = merged_df[merged_df['book_review_likes'] >= 10].reset_index(drop=True)

# Calculate like_share and popular target label
merged_df['like_share'] = merged_df['review_likes'] / merged_df['book_review_likes']
merged_df['popular'] = (merged_df['like_share'] > 0.02).astype(int)

# Calculate Days Since Review
review_dates = pd.to_datetime(merged_df['date_added'], errors='coerce', utc=True)
merged_df['days_since_review'] = (review_dates.max() - review_dates).dt.days.fillna(0)

# Difference between user rating and book average rating
merged_df['rating_diff'] = merged_df['user_rating'] - merged_df['avg_rating']

print(f"📊 Filtered dataset shape: {merged_df.shape}")
print(f"🎯 Target Class Distribution: Popular = {merged_df['popular'].mean()*100:.2f}% ({(merged_df['popular']==1).sum():,} rows), Unpopular = {(1-merged_df['popular'].mean())*100:.2f}% ({(merged_df['popular']==0).sum():,} rows)")


## Stage 2: Feature Engineering (Subsets A, B, and D)
We extract syntactic, lexical, sentiment, and TF-IDF representations.


In [ ]:
print("⏳ Extracting linguistic and sentiment features...")
sid = SentimentIntensityAnalyzer()
stopwords_set = set(nltk.corpus.stopwords.words('english'))
wnl = nltk.stem.WordNetLemmatizer()

def extract_text_features(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0, 0.0, ""
    
    quote = 1 if '"' in text else 0
    sents = nltk.sent_tokenize(text)
    num_sents = max(len(sents), 1)
    
    # Sentiment
    comp_sentiment = np.mean([sid.polarity_scores(s)['compound'] for s in sents]) if sents else 0.0
    
    # Word Tokenization
    raw_words = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    num_words = len(raw_words)
    if num_words == 0:
        return 0, 0.0, 0.0, 0.0, 0.0, 0.0, quote, comp_sentiment, ""
    
    avg_word_len = sum(len(w) for w in raw_words) / num_words
    avg_sent_len = num_words / num_sents
    
    # POS Tagging with fallback resilience
    try:
        pos_tags = nltk.pos_tag(raw_words)
    except Exception:
        try:
            nltk.download('averaged_perceptron_tagger_eng', quiet=True)
            pos_tags = nltk.pos_tag(raw_words)
        except Exception:
            pos_tags = [(w, 'VB' if w.endswith(('ing', 'ed')) else ('JJ' if w.endswith(('able', 'ful', 'ly')) else 'NN')) for w in raw_words]
    pct_verbs = sum(1 for _, tag in pos_tags if tag.startswith('V')) / num_words
    pct_nouns = sum(1 for _, tag in pos_tags if tag.startswith('N')) / num_words
    pct_adj = sum(1 for _, tag in pos_tags if tag.startswith('J') or tag.startswith('R')) / num_words
    
    # Lemmatized clean string for TF-IDF
    clean_tokens = [wnl.lemmatize(w) for w in raw_words if w not in stopwords_set]
    clean_text = " ".join(clean_tokens)
    
    return num_words, avg_word_len, avg_sent_len, pct_verbs, pct_nouns, pct_adj, quote, comp_sentiment, clean_text

# Apply feature extraction
feat_results = merged_df['review_text'].apply(extract_text_features)
feat_cols = ['num_words', 'avg_word_len', 'avg_sent_len', 'pct_verbs', 'pct_nouns', 'pct_adj', 'quote', 'sentiment', 'clean_text']
feat_df = pd.DataFrame(feat_results.tolist(), columns=feat_cols)

# Combine into master dataframe
data = pd.concat([merged_df[['popular', 'user_reviews', 'days_since_review', 'user_rating', 'rating_diff']], feat_df], axis=1)
data = data.dropna().reset_index(drop=True)
print("✅ Feature Extraction Complete!")
print(data.head(3))


## Stage 3: Train/Val/Test Splits & Class Imbalance Strategies
We create:
1. **Natural Distribution (Full Sample)**: 70% Train / 15% Val / 15% Test.
2. **Random Undersampling (Paper's Method)**: Majority class in train set trimmed down to 50:50.
3. **Cost-Sensitive Weighting (⭐ Our Add-On)**: Full data with `scale_pos_weight` in XGBoost and `class_weight='balanced'` in Logistic Regression.


In [ ]:
subset_a = ['user_reviews', 'days_since_review', 'user_rating', 'rating_diff']
subset_b = subset_a + ['num_words', 'avg_word_len', 'avg_sent_len', 'pct_verbs', 'pct_nouns', 'pct_adj', 'quote', 'sentiment']

# 70% Train, 15% Validation, 15% Test
X = data[subset_b + ['clean_text']]
y = data['popular']

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=229, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=229, stratify=y_train_val)

print(f"📊 Dataset Splits: Train={len(X_train):,}, Validation={len(X_val):,}, Test={len(X_test):,}")

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[subset_b])
X_val_scaled = scaler.transform(X_val[subset_b])
X_test_scaled = scaler.transform(X_test[subset_b])

# TF-IDF (Subset D)
tfidf = TfidfVectorizer(max_features=2000)
X_train_tfidf = tfidf.fit_transform(X_train['clean_text']).toarray()
X_test_tfidf = tfidf.transform(X_test['clean_text']).toarray()
X_train_d = np.hstack([X_train_scaled, X_train_tfidf])
X_test_d = np.hstack([X_test_scaled, X_test_tfidf])

# Strategy 2: Random Undersampled Training Set (50:50)
neg_idx = np.where(y_train == 0)[0]
pos_idx = np.where(y_train == 1)[0]
rng = np.random.default_rng(229)
dropped_neg = rng.choice(neg_idx, size=len(neg_idx) - len(pos_idx), replace=False)
keep_idx = np.setdiff1d(np.arange(len(y_train)), dropped_neg)

X_train_under = X_train_scaled[keep_idx]
y_train_under = y_train.iloc[keep_idx].values
X_train_d_under = X_train_d[keep_idx]

# Strategy 3: Cost-Sensitive Weights (⭐ Our Add-On)
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"⚖️ Calculated Cost-Sensitive Scale Pos Weight: {pos_weight:.2f}")


In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_true, y_pred)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    
    return {
        'Model & Setup': name,
        'Accuracy': round(acc, 3),
        'Sensitivity (Recall)': round(sens, 3),
        'Specificity': round(spec, 3),
        'Precision': round(prec, 3),
        'F1-Score': round(f1, 3),
        'ROC-AUC': round(auc, 3)
    }

results = []
roc_curves = {}


## Stage 4: Baseline Models vs. ⭐ Our Cost-Sensitive Add-On
We compare:
1. **Logistic Regression** (Full, Undersampled, Cost-Sensitive).
2. **XGBoost on GPU** (Full, Undersampled, Cost-Sensitive `scale_pos_weight`).
3. **PyTorch Deep Neural Network** (GPU Accelerated).


In [ ]:
print("🤖 1. Training Logistic Regression Models...")

# 1a. Baseline Full
lr_full = LogisticRegression(max_iter=1000, random_state=229)
lr_full.fit(X_train_scaled, y_train)
p_pred, p_prob = lr_full.predict(X_test_scaled), lr_full.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("Logistic Reg (Full Data - Baseline)", y_test, p_pred, p_prob))
roc_curves["Logistic Reg (Full)"] = (y_test, p_prob)

# 1b. Undersampled
lr_under = LogisticRegression(max_iter=1000, random_state=229)
lr_under.fit(X_train_under, y_train_under)
p_pred, p_prob = lr_under.predict(X_test_scaled), lr_under.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("Logistic Reg (Undersampled - Paper)", y_test, p_pred, p_prob))
roc_curves["Logistic Reg (Undersampled)"] = (y_test, p_prob)

# 1c. Cost-Sensitive
lr_cost = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=229)
lr_cost.fit(X_train_scaled, y_train)
p_pred, p_prob = lr_cost.predict(X_test_scaled), lr_cost.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("⭐ Logistic Reg (Cost-Sensitive)", y_test, p_pred, p_prob))
roc_curves["Logistic Reg (Cost-Sensitive)"] = (y_test, p_prob)

print("🤖 2. Training GPU-Accelerated XGBoost Models...")

# 2a. XGBoost Undersampled
xgb_under = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    random_state=229
)
xgb_under.fit(X_train_under, y_train_under)
p_pred, p_prob = xgb_under.predict(X_test_scaled), xgb_under.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("XGBoost (Undersampled - Paper)", y_test, p_pred, p_prob))
roc_curves["XGBoost (Undersampled)"] = (y_test, p_prob)

# 2b. ⭐ XGBoost Cost-Sensitive Add-On
xgb_cost = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=pos_weight,
    tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    random_state=229
)
xgb_cost.fit(X_train_scaled, y_train)
p_pred, p_prob = xgb_cost.predict(X_test_scaled), xgb_cost.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("⭐ XGBoost (Cost-Sensitive Add-On)", y_test, p_pred, p_prob))
roc_curves["XGBoost (Cost-Sensitive)"] = (y_test, p_prob)

# 2c. XGBoost on TF-IDF (Subset D)
xgb_d = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=pos_weight,
    tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    random_state=229
)
xgb_d.fit(X_train_d, y_train)
p_pred, p_prob = xgb_d.predict(X_test_d), xgb_d.predict_proba(X_test_d)[:, 1]
results.append(evaluate_model("⭐ XGBoost (Subset D: TF-IDF + Cost-Sensitive)", y_test, p_pred, p_prob))
roc_curves["XGBoost (TF-IDF)"] = (y_test, p_prob)


In [ ]:
print("🤖 3. Training PyTorch Deep Neural Network on GPU...")

class ReviewMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

# Prepare PyTorch Tensors
train_dataset = TensorDataset(torch.tensor(X_train_under, dtype=torch.float32), torch.tensor(y_train_under, dtype=torch.float32).unsqueeze(1))
test_dataset = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

mlp_model = ReviewMLP(len(subset_b)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# Training loop
mlp_model.train()
for epoch in range(25):
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = mlp_model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

# Evaluation
mlp_model.eval()
all_probs = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        probs = torch.sigmoid(mlp_model(batch_x)).cpu().numpy()
        all_probs.extend(probs.flatten())

all_probs = np.array(all_probs)
mlp_pred = (all_probs >= 0.5).astype(int)
results.append(evaluate_model("PyTorch MLP (GPU - Undersampled)", y_test, mlp_pred, all_probs))
roc_curves["PyTorch MLP"] = (y_test, all_probs)
print("✅ All Baseline & Add-On Models Evaluated Successfully!")


## Stage 5: 🔬 Student-Led Genuine Experimentation
Here we perform hypothesis-driven experiments across syllabus topics:
- **Experiment 1 (Unit 2)**: ANN Architecture Depth & Learning Rate Sweep.
- **Experiment 2 (Unit 1)**: Decision Tree Depth vs. Empirical Bias-Variance Tradeoff.


In [ ]:
print("🔬 Experiment 1: ANN Architecture Depth & Learning Rate Sweeps...")

architectures = {
    '1-Layer (8 units)': [8],
    '2-Layer (16->8 + Dropout 0.3)': [16, 8],
    '3-Layer (64->32->8 Overparam)': [64, 32, 8]
}

lr_sweep_results = []

for arch_name, hidden_dims in architectures.items():
    for lr in [0.01, 0.001]:
        # Build dynamic MLP
        layers = []
        in_dim = len(subset_b)
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            if len(hidden_dims) > 1:
                layers.append(nn.Dropout(0.3))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        
        dyn_model = nn.Sequential(*layers).to(device)
        dyn_opt = optim.Adam(dyn_model.parameters(), lr=lr)
        
        dyn_model.train()
        for epoch in range(15):
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                dyn_opt.zero_grad()
                loss = criterion(dyn_model(bx), by)
                loss.backward()
                dyn_opt.step()
        
        dyn_model.eval()
        p_list = []
        with torch.no_grad():
            for bx, _ in test_loader:
                bx = bx.to(device)
                p_list.extend(torch.sigmoid(dyn_model(bx)).cpu().numpy().flatten())
        p_list = np.array(p_list)
        p_pred = (p_list >= 0.5).astype(int)
        
        auc = roc_auc_score(y_test, p_list)
        f1 = f1_score(y_test, p_pred, zero_division=0)
        lr_sweep_results.append({
            'Architecture': arch_name,
            'Learning Rate': lr,
            'ROC-AUC': round(auc, 3),
            'F1-Score': round(f1, 3)
        })

print("📊 ANN Architecture & Learning Rate Ablation Table:")
print(pd.DataFrame(lr_sweep_results))

print("\n🔬 Experiment 2: Decision Tree Depth vs. Bias-Variance Curve...")
depths = range(1, 16)
train_scores = []
test_scores = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=229)
    dt.fit(X_train_under, y_train_under)
    train_scores.append(roc_auc_score(y_train_under, dt.predict_proba(X_train_under)[:, 1]))
    test_scores.append(roc_auc_score(y_test, dt.predict_proba(X_test_scaled)[:, 1]))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, 'o-', label='Training ROC-AUC', color='crimson')
plt.plot(depths, test_scores, 's-', label='Validation/Test ROC-AUC', color='royalblue')
plt.title('Empirical Bias-Variance Tradeoff: Decision Tree Max Depth Sweep', fontsize=13, fontweight='bold')
plt.xlabel('Tree Max Depth', fontsize=11)
plt.ylabel('ROC-AUC Score', fontsize=11)
plt.axvline(x=4, color='green', linestyle='--', label='Optimal Depth (d=4)')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('bias_variance_curve.png', dpi=300)
plt.show()


## Stage 6: Comparative Results & Visualizations
Master comparison table summarizing all models, our Add-On, and ROC curves.


In [ ]:
results_df = pd.DataFrame(results)
print("="*85)
print("                       MASTER MODEL EVALUATION RESULTS TABLE")
print("="*85)
print(results_df.to_string(index=False))
print("="*85)

# Plot ROC Curves
plt.figure(figsize=(10, 7))
for name, (y_true, y_prob) in roc_curves.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random Chance (AUC = 0.500)')
plt.title('Receiver Operating Characteristic (ROC) Curves Comparison', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('roc_curves_comparison.png', dpi=300)
plt.show()

# Feature Importance from XGBoost
plt.figure(figsize=(9, 5))
feat_imp = pd.Series(xgb_cost.feature_importances_, index=subset_b).sort_values()
feat_imp.plot(kind='barh', color='teal')
plt.title('Feature Importance from Cost-Sensitive XGBoost (Gain Weight)', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance Score', fontsize=11)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()


## Stage 7: Interactive Popularity Predictor (Live Demo Function)
Use this interactive tester during your viva or presentation demonstration!


In [ ]:
def predict_review_popularity(review_text, user_reviews=15, days_since_review=120, user_rating=5.0, avg_book_rating=4.1):
    """
    Interactive predictor demonstrating real-time model inference.
    """
    rating_diff = user_rating - avg_book_rating
    num_words, avg_word_len, avg_sent_len, pct_verbs, pct_nouns, pct_adj, quote, sentiment, _ = extract_text_features(review_text)
    
    raw_feats = np.array([[user_reviews, days_since_review, user_rating, rating_diff,
                           num_words, avg_word_len, avg_sent_len, pct_verbs, pct_nouns, pct_adj, quote, sentiment]])
    scaled_feats = scaler.transform(raw_feats)
    
    prob_lr = lr_cost.predict_proba(scaled_feats)[0, 1]
    prob_xgb = xgb_cost.predict_proba(scaled_feats)[0, 1]
    
    print("="*60)
    print("           BOOK REVIEW POPULARITY PREDICTION")
    print("="*60)
    print(f"📝 Review Text: \"{review_text[:100]}...\"")
    print(f"👤 Reviewer Metadata: Total Reviews={user_reviews}, Rating={user_rating}★ (Diff={rating_diff:+.2f})")
    print(f"📊 Linguistic Stats: Words={num_words}, Sentiment Score={sentiment:+.3f}, Has Quote={'Yes' if quote else 'No'}")
    print("-" * 60)
    print(f"🎯 Logistic Regression Popularity Prob: {prob_lr*100:.1f}% -> {'🔥 POPULAR' if prob_lr >= 0.5 else '💤 UNPOPULAR'}")
    print(f"🎯 XGBoost Popularity Prob:           {prob_xgb*100:.1f}% -> {'🔥 POPULAR' if prob_xgb >= 0.5 else '💤 UNPOPULAR'}")
    print("="*60)

# Sample Test
sample_review = "This book completely blew my mind! The character development was phenomenal and I couldn't put it down. Highly recommended for fantasy lovers."
predict_review_popularity(sample_review, user_reviews=45, days_since_review=200, user_rating=5.0, avg_book_rating=3.9)
